# SB3 Altitude-Hold Benchmark
**Notebook version:** `2.0.0` (2024-09-10)

**Goal:** Train all five continuous-action Stable-Baselines3 algorithms — **PPO, A2C, SAC, TD3, DDPG** — on
`AltitudeHoldEnv` and compare convergence, final performance, and trajectory quality.

**Physics:** Crazyflie 2.1 simulator (RK4, full aerodynamics — `hover_env.py v2.0.0`).
**Task:** Reach and hold a horizontal altitude plane at 5 m. XY drift unconstrained.

| Section | Content |
|---------|---------|
| 1 | Installation & imports |
| 2 | Environment sanity check |
| 3 | Algorithm hyperparameter configs (PPO / A2C / SAC / TD3 / DDPG) |
| 4 | Training loop |
| 5 | Evaluation & trajectory collection |
| 6 | Professional visualisation dashboard |
| 7 | Summary & wrap-up |


## 1. Installation


In [ ]:
!pip install -q 'stable-baselines3[extra]>=2.0' gymnasium scipy


## 2. Setup: Paths & Environment


In [ ]:
import os, sys, time
import numpy as np

# -- Google Drive (Colab) or local fallback --
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/altitude_hold_rl'
except Exception:
    PROJECT_DIR = os.path.abspath('.')

CHECKPOINT_DIR  = os.path.join(PROJECT_DIR, 'checkpoints')
TENSORBOARD_DIR = os.path.join(PROJECT_DIR, 'tb_logs')
RESULTS_DIR     = os.path.join(PROJECT_DIR, 'results')
for d in [CHECKPOINT_DIR, TENSORBOARD_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Project dir:', PROJECT_DIR)


In [ ]:
# Upload hover_env.py (Colab only -- skip if already present)
try:
    from google.colab import files
    import shutil
    uploaded = files.upload()
    if 'hover_env.py' in uploaded:
        shutil.copy('hover_env.py', os.path.join(PROJECT_DIR, 'hover_env.py'))
        print('Copied hover_env.py to', PROJECT_DIR)
except Exception:
    print('Not in Colab or already uploaded -- continuing.')


In [ ]:
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from hover_env import AltitudeHoldEnv, AltitudeHoldRewardConfig
from hover_env import __version__ as ENV_VERSION
print(f'AltitudeHoldEnv v{ENV_VERSION} imported OK')


In [ ]:
from stable_baselines3.common.env_checker import check_env

_env = AltitudeHoldEnv(seed=0)
check_env(_env, warn=True)
print('check_env passed.')
print(f'  obs  shape : {_env.observation_space.shape}')
print(f'  act  shape : {_env.action_space.shape}')
del _env


## 3. Algorithm Configs


In [ ]:
# ===== Training budget =====
# Adjust to your compute. On a Colab T4:
#   ON_POLICY_STEPS=2M  -> ~30-60 min per algo
#   OFF_POLICY_STEPS=1M -> ~20-40 min per algo
ON_POLICY_STEPS  = 2_000_000
OFF_POLICY_STEPS = 1_000_000
N_CHECKPOINTS    = 8
N_PARALLEL       = 4   # parallel envs for on-policy algos

# ===== Visual palette =====
MODEL_COLORS = {
    'PPO':  '#4C72B0',   # steel blue
    'A2C':  '#55A868',   # muted green
    'SAC':  '#DD8452',   # warm orange
    'TD3':  '#C44E52',   # muted red
    'DDPG': '#8172B2',   # soft purple
}


In [ ]:
from stable_baselines3 import PPO, A2C, SAC, TD3, DDPG
from stable_baselines3.common.noise import NormalActionNoise

ALGO_CONFIGS = {
    'PPO': dict(
        cls=PPO, policy='MlpPolicy', n_envs=N_PARALLEL,
        total_steps=ON_POLICY_STEPS,
        kwargs=dict(n_steps=2048, batch_size=256, gamma=0.99,
                    learning_rate=3e-4, gae_lambda=0.95,
                    clip_range=0.2, ent_coef=0.01, n_epochs=10, verbose=1)),
    'A2C': dict(
        cls=A2C, policy='MlpPolicy', n_envs=N_PARALLEL,
        total_steps=ON_POLICY_STEPS,
        kwargs=dict(n_steps=128, gamma=0.99, learning_rate=7e-4,
                    gae_lambda=0.95, ent_coef=0.0, vf_coef=0.5,
                    max_grad_norm=0.5, verbose=1)),
    'SAC': dict(
        cls=SAC, policy='MlpPolicy', n_envs=1,
        total_steps=OFF_POLICY_STEPS,
        kwargs=dict(learning_rate=3e-4, buffer_size=500_000,
                    batch_size=256, gamma=0.99, tau=0.005,
                    ent_coef='auto', learning_starts=10_000, verbose=1)),
    'TD3': dict(
        cls=TD3, policy='MlpPolicy', n_envs=1,
        total_steps=OFF_POLICY_STEPS,
        kwargs=dict(learning_rate=1e-3, buffer_size=500_000,
                    batch_size=256, gamma=0.99, tau=0.005,
                    train_freq=1, learning_starts=10_000,
                    action_noise=None, verbose=1)),
    'DDPG': dict(
        cls=DDPG, policy='MlpPolicy', n_envs=1,
        total_steps=OFF_POLICY_STEPS,
        kwargs=dict(learning_rate=1e-3, buffer_size=300_000,
                    batch_size=256, gamma=0.99, tau=0.005,
                    train_freq=1, learning_starts=10_000,
                    action_noise=None, verbose=1)),
}
print(f'Configured {len(ALGO_CONFIGS)} algorithms:', list(ALGO_CONFIGS.keys()))


## 4. Training


In [ ]:
from stable_baselines3.common.callbacks import (
    BaseCallback, CheckpointCallback, CallbackList)
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv


class RewardComponentCallback(BaseCallback):
    """Logs per-step reward components to TensorBoard once per rollout."""
    KEYS = ['r_alive','r_alt','r_tilt','r_angvel',
            'r_thrust','r_smooth','alt_err','tilt','angvel_norm']
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self._buf = {k: [] for k in self.KEYS}
    def _on_step(self):
        for info in self.locals.get('infos', []):
            for k in self.KEYS:
                if k in info:
                    self._buf[k].append(float(info[k]))
        return True
    def _on_rollout_end(self):
        for k, vals in self._buf.items():
            if vals:
                self.logger.record(f'reward_components/{k}', float(np.mean(vals)))
            self._buf[k] = []


def _make_env(monitor_dir, env_id=0, **kw):
    """Factory for Monitor-wrapped AltitudeHoldEnv.
    monitor_dir: where Monitor CSV will be written (one per env)."""
    def _init():
        env = AltitudeHoldEnv(**kw)
        mon_path = os.path.join(monitor_dir, f'env_{env_id}')
        return Monitor(env, filename=mon_path)
    return _init


def train_one(name, cfg):
    """Train one SB3 algorithm. Returns (model, final_save_path)."""
    n_envs       = cfg['n_envs']
    total        = cfg['total_steps']
    AlgoCls      = cfg['cls']
    kwargs       = dict(cfg['kwargs'])
    ckpt_dir     = os.path.join(CHECKPOINT_DIR, name)
    tb_dir       = os.path.join(TENSORBOARD_DIR, name)
    monitor_dir  = os.path.join(ckpt_dir, 'monitor')
    os.makedirs(ckpt_dir,    exist_ok=True)
    os.makedirs(tb_dir,      exist_ok=True)
    os.makedirs(monitor_dir, exist_ok=True)

    vec_env = DummyVecEnv([_make_env(monitor_dir, env_id=i)
                           for i in range(n_envs)])

    # Inject action noise for TD3 / DDPG
    if 'action_noise' in kwargs and kwargs['action_noise'] is None:
        if name in ('TD3', 'DDPG'):
            n_act = vec_env.action_space.shape[0]
            kwargs['action_noise'] = NormalActionNoise(
                np.zeros(n_act), 0.1*np.ones(n_act))
        else:
            del kwargs['action_noise']

    model = AlgoCls(cfg['policy'], vec_env,
                    tensorboard_log=tb_dir, **kwargs)

    save_freq = max(1, (total // N_CHECKPOINTS) // n_envs)
    cb = CallbackList([
        CheckpointCallback(save_freq=save_freq, save_path=ckpt_dir,
                           name_prefix=name.lower()),
        RewardComponentCallback(),
    ])

    sep = '=' * 55
    print(f'\n{sep}')
    print(f'  Training {name}  |  {total:,} steps  |  {n_envs} env(s)')
    print(f'{sep}')
    t0 = time.time()
    model.learn(total_timesteps=total, callback=cb, progress_bar=True)
    elapsed_min = (time.time() - t0) / 60
    print(f'  Done in {elapsed_min:.1f} min')

    final_path = os.path.join(ckpt_dir, f'{name.lower()}_final')
    model.save(final_path)
    print(f'  Saved final: {final_path}.zip')
    vec_env.close()
    return model, final_path


In [ ]:
TRAINED = {}
FINAL_PATHS = {}

for algo_name, cfg in ALGO_CONFIGS.items():
    model, fpath = train_one(algo_name, cfg)
    TRAINED[algo_name]     = model
    FINAL_PATHS[algo_name] = fpath

print('\nAll algorithms trained!')


In [ ]:
# Optional: launch TensorBoard for live monitoring
%load_ext tensorboard
%tensorboard --logdir {TENSORBOARD_DIR}


## 5. Evaluation -- Collect Trajectories


In [ ]:
import glob, re
_CLS = {'PPO':PPO, 'A2C':A2C, 'SAC':SAC, 'TD3':TD3, 'DDPG':DDPG}

def rollout(model_path, algo, n_steps=1000, seed=0):
    """Load a saved model and run one deterministic evaluation episode."""
    pol  = _CLS[algo].load(model_path)
    env  = AltitudeHoldEnv(seed=seed)
    obs, _ = env.reset(seed=seed)
    pos, rewards, alt_errs, tilts = [env._p_WB.copy()], [], [], []
    crashed = oob = False
    for _ in range(n_steps):
        action, _ = pol.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = env.step(action)
        pos.append(env._p_WB.copy())
        rewards.append(float(r))
        alt_errs.append(info['alt_err'])
        tilts.append(info['tilt'])
        if info.get('crashed'): crashed = True
        if info.get('out_of_bounds'): oob = True
        if terminated or truncated: break
    return dict(
        algo=algo, model_path=str(model_path),
        p_WB=np.array(pos),
        rewards=rewards, alt_errs=alt_errs, tilts=tilts,
        ep_reward=float(sum(rewards)),
        ep_len=len(rewards),
        mean_alt_err=float(np.mean(alt_errs)) if alt_errs else float('nan'),
        mean_tilt_deg=float(np.degrees(np.mean(tilts))) if tilts else 0.,
        target_alt=env.target_alt,
        crashed=crashed, oob=oob,
    )


def collect_checkpoints(name):
    """Collect rollouts from every checkpoint + final model."""
    pat   = os.path.join(CHECKPOINT_DIR, name, f'{name.lower()}_*_steps.zip')
    paths = sorted(
        glob.glob(pat),
        key=lambda p: int(re.search(r'_(\d+)_steps', p).group(1))
                      if re.search(r'_(\d+)_steps', p) else -1,
    )
    fz = FINAL_PATHS.get(name, '') + '.zip'
    if os.path.exists(fz) and fz not in paths:
        paths.append(fz)
    print(f'  {name}: {len(paths)} checkpoints')
    return [rollout(p.replace('.zip',''), name) for p in paths]


In [ ]:
print('Collecting checkpoint rollouts...')
ALL_ROLLOUTS = {n: collect_checkpoints(n) for n in ALGO_CONFIGS}

print('\nResults summary:')
for name, rs in ALL_ROLLOUTS.items():
    if rs:
        b = rs[-1]
        print(f"  {name:5s}  ckpts={len(rs)}  "
              f"reward={b['ep_reward']:8.1f}  "
              f"alt_err={b['mean_alt_err']:.3f}m  "
              f"ep_len={b['ep_len']}")


## 6. Visualisation Dashboard


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_rgba, LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F5F5F5',
    'axes.edgecolor':   '#CCCCCC',
    'axes.grid':        True,
    'grid.color':       '#DDDDDD',
    'grid.linewidth':   0.7,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#CCCCCC',
    'legend.fontsize':  10,
    'lines.linewidth':  1.8,
})
print('Matplotlib style configured.')


### 6a. Convergence Curves


In [ ]:
from stable_baselines3.common.results_plotter import load_results, ts2xy
import pandas as pd

fig_conv, (ax_r, ax_l) = plt.subplots(1, 2, figsize=(16, 5))
fig_conv.suptitle('Convergence -- All Algorithms', fontsize=15, fontweight='bold')

def fmt_steps(x, _):
    return f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'

for name, color in MODEL_COLORS.items():
    # Monitor CSVs live in CHECKPOINT_DIR/name/monitor/
    monitor_dir = os.path.join(CHECKPOINT_DIR, name, 'monitor')
    try:
        res = load_results(monitor_dir)
        x, y = ts2xy(res, 'timesteps')
        y_sm = pd.Series(y).rolling(20, min_periods=1).mean().values
        ax_r.plot(x, y,    color=color, alpha=0.15, linewidth=0.8)
        ax_r.plot(x, y_sm, color=color, linewidth=2.2, label=name)
        el   = pd.Series(res['l'].values).rolling(20, min_periods=1).mean().values
        ax_l.plot(x[:len(el)], el, color=color, linewidth=2.2, label=name)
    except Exception as e:
        print(f'  [{name}] no monitor data: {e}')

ax_r.axhline(0, color='#888', linestyle='--', linewidth=1, alpha=0.6)
ax_r.set_title('Episode Reward')
ax_r.set_xlabel('Timesteps')
ax_r.set_ylabel('Episode Reward')
ax_r.legend()
ax_r.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_steps))
ax_l.set_title('Episode Length (steps)')
ax_l.set_xlabel('Timesteps')
ax_l.set_ylabel('Steps')
ax_l.set_ylim(0, 1050)
ax_l.legend()
ax_l.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_steps))
fig_conv.tight_layout()
fig_conv.savefig(os.path.join(RESULTS_DIR, '1_convergence.png'),
                 dpi=150, bbox_inches='tight')
plt.show()


### 6b. 3D Trajectory Gallery


In [ ]:
def algo_cmap(hex_color):
    """Create a light->dark colourmap for checkpoint progression."""
    r0, g0, b0, _ = to_rgba('#DDDDDD')
    r1, g1, b1, _ = to_rgba(hex_color)
    return LinearSegmentedColormap.from_list('ck', [(r0,g0,b0), (r1,g1,b1)])

n_algos = len(ALGO_CONFIGS)
n_cols  = 3
n_rows  = (n_algos + n_cols - 1) // n_cols

fig_t = plt.figure(figsize=(6*n_cols, 5.5*n_rows))
fig_t.suptitle(
    '3D Trajectory Gallery\n'
    'Light trail = Early training    Dark trail = Final checkpoint    Gold plane = Target altitude',
    fontsize=13, fontweight='bold', y=1.01)

for idx, (name, rollouts) in enumerate(ALL_ROLLOUTS.items()):
    ax   = fig_t.add_subplot(n_rows, n_cols, idx+1, projection='3d')
    cmap = algo_cmap(MODEL_COLORS[name])
    nck  = max(1, len(rollouts))
    for i, r in enumerate(rollouts):
        t     = (i + 1) / nck
        p     = r['p_WB']
        if len(p) < 2:
            continue
        color = cmap(t)
        alpha = float(0.20 + 0.80 * t)
        lw    = 1.0 + 1.2 * t
        ax.plot(p[:,0], p[:,1], p[:,2],
                color=color, alpha=alpha, linewidth=lw)
        ax.scatter([p[0,0]], [p[0,1]], [p[0,2]], c=[color],
                   s=18, marker='o', alpha=alpha, depthshade=False)
        ax.scatter([p[-1,0]], [p[-1,1]], [p[-1,2]], c=[color],
                   s=50, marker='*', alpha=alpha, depthshade=False)
    # Target altitude plane
    if rollouts:
        tz   = rollouts[-1]['target_alt']
        xx, yy = np.meshgrid(np.linspace(-2, 2, 4), np.linspace(-2, 2, 4))
        ax.plot_surface(xx, yy, np.full_like(xx, tz), alpha=0.12, color='gold')
    ax.set_title(f'{name}\n({nck} checkpoints)',
                 fontweight='bold', color=MODEL_COLORS[name], pad=3)
    ax.set_xlabel('X (m)', fontsize=8)
    ax.set_ylabel('Y (m)', fontsize=8)
    ax.set_zlabel('Z (m)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.view_init(elev=25, azim=45)
    # Colourbar = training stage legend
    sm = ScalarMappable(cmap=cmap, norm=Normalize(0, 1))
    sm.set_array([])
    cb = fig_t.colorbar(sm, ax=ax, pad=0.12, shrink=0.55, aspect=12)
    cb.set_label('Training Stage', fontsize=8)
    cb.set_ticks([0, 0.5, 1.0])
    cb.set_ticklabels(['Early', 'Mid', 'Final'], fontsize=7)
fig_t.tight_layout()
fig_t.savefig(os.path.join(RESULTS_DIR, '2_trajectory_gallery.png'),
              dpi=150, bbox_inches='tight')
plt.show()


### 6c. Radar Chart


In [ ]:
def radar(data, cats, title, colors):
    """Draw a radar/spider chart.  data: {name: [v1,..,vN]} all in [0,1]."""
    N = len(cats)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig_r, ax_r = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})
    fig_r.patch.set_facecolor('#FAFAFA')
    ax_r.set_facecolor('#F0F0F0')
    ax_r.set_theta_offset(np.pi / 2)
    ax_r.set_theta_direction(-1)
    ax_r.set_ylim(0, 1)
    ax_r.set_yticks([.25, .5, .75, 1])
    ax_r.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(cats, fontsize=11, fontweight='bold')
    handles = []
    for name, vals in data.items():
        v = vals + vals[:1]
        c = colors[name]
        ax_r.plot(angles, v, lw=2.5, color=c)
        ax_r.fill(angles, v, alpha=0.12, color=c)
        handles.append(mpatches.Patch(color=c, label=name))
    ax_r.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax_r.legend(handles=handles, loc='upper right',
                bbox_to_anchor=(1.4, 1.15))
    return fig_r

cats = ['Episode\nReward', 'Episode\nLength', 'Alt Err\n(lower=better)',
        'Tilt\n(lower=better)', 'Survival']

fm = {}
for name in ALGO_CONFIGS:
    rs = ALL_ROLLOUTS.get(name, [])
    if not rs:
        continue
    r = rs[-1]
    fm[name] = [
        r['ep_reward'], r['ep_len'],
        r['mean_alt_err'], r['mean_tilt_deg'],
        float(not r['crashed']),
    ]

if fm:
    arr  = np.array([fm[n] for n in fm])
    lo   = arr.min(0)
    hi   = arr.max(0)
    rng  = np.where((hi - lo) < 1e-9, 1., hi - lo)
    norm = (arr - lo) / rng
    norm[:, 2] = 1. - norm[:, 2]   # flip: lower alt_err = better
    norm[:, 3] = 1. - norm[:, 3]   # flip: lower tilt = better
    rd = {n: norm[i].tolist() for i, n in enumerate(fm)}
    fig_rad = radar(rd, cats,
        'Final-Checkpoint Performance\n(outer edge = better)', MODEL_COLORS)
    fig_rad.savefig(os.path.join(RESULTS_DIR, '3_radar.png'),
                    dpi=150, bbox_inches='tight')
    plt.show()


### 6d. Summary Table


In [ ]:
import pandas as pd

rows = []
for name in ALGO_CONFIGS:
    rs = ALL_ROLLOUTS.get(name, [])
    if not rs:
        continue
    r = rs[-1]
    rows.append({
        'Algorithm':    name,
        'Steps':        f"{ALGO_CONFIGS[name]['total_steps']:,}",
        'Ep Reward':    f"{r['ep_reward']:.1f}",
        'Ep Length':    str(r['ep_len']),
        'Alt Err (m)':  f"{r['mean_alt_err']:.3f}",
        'Tilt (deg)':   f"{r['mean_tilt_deg']:.2f}",
        'Crashed':      'YES' if r['crashed'] else 'no',
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

fig_tb, ax_tb = plt.subplots(figsize=(14, max(2, 0.55*len(rows)+1.5)))
ax_tb.axis('off')
tbl = ax_tb.table(cellText=df.values, colLabels=df.columns,
                  cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.65)
for j in range(len(df.columns)):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i, name in enumerate([row['Algorithm'] for row in rows]):
    tbl[i+1, 0].set_facecolor(MODEL_COLORS.get(name, '#FFFFFF'))
    tbl[i+1, 0].set_text_props(color='white', fontweight='bold')
fig_tb.suptitle('Algorithm Comparison Summary', fontsize=14, fontweight='bold')
fig_tb.tight_layout()
fig_tb.savefig(os.path.join(RESULTS_DIR, '4_summary_table.png'),
               dpi=150, bbox_inches='tight')
plt.show()


### 6e. Best Algorithm -- Altitude Profile Deep Dive


In [ ]:
best = max(
    (n for n in ALL_ROLLOUTS if ALL_ROLLOUTS[n]),
    key=lambda n: ALL_ROLLOUTS[n][-1]['ep_reward'],
)
br = ALL_ROLLOUTS[best][-1]
print(f'Best: {best}  reward={br["ep_reward"]:.1f}  '
      f'ep_len={br["ep_len"]}  alt_err={br["mean_alt_err"]:.3f} m')

fig_z, (az, ar) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig_z.suptitle(f'Best Algorithm: {best} -- Altitude Hold Profile',
               fontsize=14, fontweight='bold')
col = MODEL_COLORS[best]
p   = br['p_WB']
ts  = np.arange(len(p)) * 0.01
tz  = br['target_alt']

az.plot(ts, p[:, 2], color=col, lw=2.0, label='Drone altitude')
az.axhline(tz, color='gold', lw=2.0, linestyle='--',
           label=f'Target: {tz:.1f} m')
az.fill_between(ts, tz - 0.3, tz + 0.3, alpha=0.12, color='gold',
                label='+-0.3 m band')
az.set_ylabel('Altitude (m)')
az.set_title('Altitude vs. Time')
az.legend()

rts = np.arange(len(br['rewards'])) * 0.01
ar.plot(rts, np.cumsum(br['rewards']), color=col, lw=2.0)
ar.set_ylabel('Cumulative Reward')
ar.set_xlabel('Time (s)')
ar.set_title('Cumulative Reward vs. Time')

fig_z.tight_layout()
fig_z.savefig(os.path.join(RESULTS_DIR, f'5_best_{best}.png'),
              dpi=150, bbox_inches='tight')
plt.show()


## 7. Save & Wrap-Up


In [ ]:
import shutil
print('Results saved to:', RESULTS_DIR)
for f in sorted(os.listdir(RESULTS_DIR)):
    sz = os.path.getsize(os.path.join(RESULTS_DIR, f)) // 1024
    print(f'  {f:50s}  {sz:5d} KB')

best_src = FINAL_PATHS.get(best, '') + '.zip'
if os.path.exists(best_src):
    shutil.copy(best_src, os.path.join(RESULTS_DIR, f'best_{best}.zip'))
    print(f'\nBest model ({best}) copied to results.')

print('\nDone!')
print(f'Environment : hover_env.py v{ENV_VERSION}')
print(f'Notebook    : SB3_AltitudeHold_Benchmark.ipynb v2.0.0')
